# Overfitting und Underfitting

Overfitting und Underfitting kennst du bereits von klassischen Machine-Learning-Algorithmen. Beim Training eines neuronalen Netzes spielen sie eine genauso große Rolle.

In früheren Notebooks hast du vielleicht gesehen, wie die Validierungs-Accuracy (oder der MAE) nach einer Reihe von Epochen einen Höchstwert (oder Tiefstwert) erreicht und dann stagniert oder schlechter wird, während sich die Trainingsmetrik weiter verbessert. Diese Lücke ist das klassische Zeichen für **Overfitting**: Das Modell passt sich so eng an die Trainingsdaten an, dass es nicht mehr auf ungesehene Daten generalisiert.

**Underfitting** ist das umgekehrte Problem. Ein Modell, das zu klein ist, zu stark regularisiert ist oder zu wenige Epochen lang trainiert wurde, kann die relevanten Muster nicht erfassen. Deshalb schneidet es sowohl auf dem Trainings- als auch auf dem Testdatensatz schlecht ab.

Der beste Schutz gegen Overfitting sind mehr und bessere Trainingsdaten, die das gesamte Spektrum der Eingaben abdecken. Wenn es die nicht gibt, helfen Regularisierungstechniken. Dieses Notebook erkundet die gängigsten davon am Datensatz Boston Housing.

## Lernziele

Am Ende dieses Notebooks kannst du:

- Overfitting und Underfitting anhand von Trainings- und Validierungskurven diagnostizieren.
- neuronale Netze mit unterschiedlicher Kapazität auf demselben Datensatz erstellen und vergleichen.
- L1- und L2-Gewichtsregularisierung sowie Dropout anwenden, um Overfitting zu verringern.
- die Modelle auf einem zurückgehaltenen Testdatensatz evaluieren und vergleichen.

## Setup

Importiere vor dem Start die nötigen Pakete:

In [ ]:
import os

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

import tensorflow as tf

from tensorflow.keras import regularizers


In [ ]:
import tensorflow_docs as tfdocs
import tensorflow_docs.modeling
import tensorflow_docs.plots

In [ ]:
# Seeds usw. definieren
RSEED = 42
tf.keras.backend.set_floatx("float64")
plt.rcParams["figure.figsize"] = (15, 10)

## Der Datensatz Boston Housing

In diesem Notebook verwenden wir den Datensatz Boston Housing. Du findest ihn im Ordner data in diesem Repo.

In den nächsten Zellen werden wir:
- die CSV-Datei in einen pandas-DataFrame laden und einen kurzen Blick auf Form und Spalten werfen,
- auf fehlende Werte prüfen, damit wir wissen, ob eine Imputation nötig ist,
- die Zielspalte (Median des Hauswerts) aus dem Feature-DataFrame herauslösen,
- eine Aufteilung in Trainings- und Testdaten erstellen, um wirklich ungesehene Daten zurückzuhalten,
- einen `StandardScaler` nur auf den Trainings-Features fitten und auf beide Teile anwenden, um Data Leakage zu vermeiden,
- die skalierten Features und die Zielvariable in die NumPy-Arrays umwandeln, die Keras erwartet.

Eine deterministische Vorverarbeitung (Aufteilung mit festem Seed + Scaler nur auf den Trainingsdaten gefittet) macht die nachgelagerten Modellvergleiche fair.

In [ ]:
# Daten aus dem Ordner data laden
df = pd.read_csv("../data/boston.csv")
df.head(3)

In [ ]:
# Form des DataFrames
df.shape

In [ ]:
# Auf fehlende Werte prüfen
df.isnull().sum()

In [ ]:
# Zielvariable definieren und aus df entfernen
target = df.pop("MEDV")

In [ ]:
# In Trainings- und Testdatensatz aufteilen
X_train, X_test, y_train, y_test = train_test_split(df, target, random_state=RSEED)

In [ ]:
# Numerische Werte skalieren
col_scale = [
    "CRIM",
    "ZN",
    "INDUS",
    "NOX",
    "RM",
    "AGE",
    "DIS",
    "TAX",
    "PTRATIO",
    "LSTAT",
]

scaler = StandardScaler()
X_train[col_scale] = scaler.fit_transform(X_train[col_scale])
X_test[col_scale] = scaler.transform(X_test[col_scale])

In [ ]:
# In np-Array umwandeln
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

## Modellkomplexität

Eine Möglichkeit, Under- oder Overfitting ohne Regularisierungstechnik entgegenzuwirken, ist eine Anpassung der Modellkomplexität. Je größer dein neuronales Netz ist, desto leichter kommt es zu Overfitting auf den Daten.

Leider gibt es keine magische Formel, um die richtige Größe oder Architektur deines Modells zu bestimmen (was die Anzahl der Schichten oder die richtige Größe jeder Schicht angeht). Du musst mit einer Reihe verschiedener Architekturen experimentieren.
Um eine passende Modellgröße zu finden, beginnst du am besten mit relativ wenigen Schichten und Parametern. Dann vergrößerst du die Schichten oder fügst neue Schichten hinzu, bis der Validierungsverlust schlechter wird.

In diesem Notebook trainieren wir drei verschiedene Modelle mit unterschiedlicher Komplexität auf dem Datensatz Boston Housing und vergleichen ihre Leistung. Die Ergebnisse speichern wir in einem Dictionary, und die folgenden Funktionen zur Darstellung helfen uns, die Leistung zu visualisieren.
Außerdem definieren wir den Optimierer, den wir verwenden wollen, sowie eine [Abnahme der Lernrate](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/schedules/InverseTimeDecay). Viele Modelle trainieren besser, wenn die Lernrate während des Trainings allmählich sinkt.
Mit `optimizers.schedule` können wir die Lernrate im Lauf der Zeit verringern:

Zusätzlich sammeln wir jeden Trainingsdurchlauf im Dictionary `training_history`. Jeder Eintrag speichert das `History`-Objekt, das `model.fit` zurückgibt. Es enthält pro Epoche `loss` und `mean_squared_error` für den Trainings- und den Validierungsdatensatz. Die Hilfsfunktionen zur Darstellung weiter unten lesen einfach diese Logs. Solange die Schlüsselnamen gleich bleiben, kannst du also neue Modellvarianten hinzufügen und bekommst automatisch vergleichbare Diagramme.

In [ ]:
# Dictionaries zum Speichern der Ergebnisse definieren
training_history = {}
evaluation_results = {}

# Anzahl der Epochen und Abnahme der Lernrate definieren
N_TRAIN = len(X_train)
EPOCHS = 2000
BATCH_SIZE = 32
STEPS_PER_EPOCH = N_TRAIN // BATCH_SIZE
lr_schedule = tf.keras.optimizers.schedules.InverseTimeDecay(
    0.01, decay_steps=STEPS_PER_EPOCH * 1000, decay_rate=1, staircase=False
)


# Optimierer für die Modellierung definieren
def make_optimizer():
    # Keras 3 bindet einen Optimierer an die Variablen eines Modells, daher pro Modell eine neue Instanz erstellen
    return tf.keras.optimizers.Adam(learning_rate=lr_schedule, name="Adam")

In [ ]:
# Funktion zur Darstellung des MSE
def plot_metric(history):
    plt.plot(history.history["mse"])
    plt.plot(history.history["val_mse"])
    plt.title("Model MSE")
    plt.ylabel("MSE")
    plt.xlabel("Epoch")
    plt.legend(["train", "validation"], loc="upper right")
    plt.show()

In [ ]:
# Funktion zur Darstellung des Verlusts
def plot_loss(history):
    plt.plot(history.history["loss"], label="loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.title("Model Loss")
    plt.ylim([0, 10])
    plt.xlabel("Epoch")
    plt.ylabel("Error")
    plt.legend()
    plt.grid(True)

### Kleines Modell

Wir beginnen mit einem eher kleinen Modell mit zwei verborgenen Schichten und 32 Neuronen.

Die folgende Funktion enthält die Architektur unseres Modells (die Anzahl der Schichten und Neuronen) sowie Informationen zum Optimierer, zum Verlust und zur Metrik, die wir verwenden.

Wenn du `small_model.fit(...)` aufrufst, achte darauf, wie schnell die Validierungskurven abflachen. Bleiben Trainings- und Validierungs-MSE beide hoch, zeigt das Netz Underfitting. Sinkt der Trainings-MSE weiter, während der Validierungs-MSE einen Tiefpunkt erreicht und wieder steigt, kommst du in den Bereich des Overfittings. Dieser Baseline-Durchlauf gibt uns einen Referenzpunkt für die größeren Architekturen.

In [ ]:
# Modellarchitektur in einer Funktion definieren
def get_compiled_small_model():
    small_model = tf.keras.Sequential(
        [
            tf.keras.Input(shape=(12,)),
            tf.keras.layers.Dense(
                32, kernel_initializer="uniform", activation="relu"
            ),
            tf.keras.layers.Dense(32, kernel_initializer="uniform", activation="relu"),
            tf.keras.layers.Dense(1, kernel_initializer="uniform"),
        ]
    )

    small_model.compile(optimizer=make_optimizer(), loss="mae", metrics=["mse"])
    return small_model

In [ ]:
# Kleines Modell instanziieren und Zusammenfassung des Modells ausgeben
with tf.device("/cpu:0"):
    small_model = get_compiled_small_model()
    print(small_model.summary())

In [ ]:
with tf.device("/cpu:0"):
    training_history["small"] = small_model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        verbose=0,
        steps_per_epoch=STEPS_PER_EPOCH,
        epochs=EPOCHS,
    )

Wir können den Fortschritt visualisieren, den unser Modell während des Trainings macht, indem wir den MSE auf dem Trainings- und dem Validierungsdatensatz in Abhängigkeit von der Anzahl der Epochen darstellen.

Eine wachsende Lücke zwischen der Trainings- und der Validierungskurve des MSE ist das klassische Zeichen für Overfitting. Liegen die Kurven übereinander, aber auf einem hohen Wert, ist das Modell zu einfach oder zu wenig trainiert.

In [ ]:
# MSE-Verlauf darstellen
plot_metric(training_history["small"])

Dasselbe funktioniert für den Verlust. Wir können ebenso visualisieren, wie sich der Verlust während des Trainings verbessert hat:

In [ ]:
# Verlauf des Verlusts darstellen
plot_loss(training_history["small"])

Wenn wir die Leistung unseres Modells auf ungesehenen Daten evaluieren wollen, können wir den zurückgehaltenen Testdatensatz verwenden, den wir zu Beginn der Modellierung beiseitegelegt haben. Du kannst die Leistung des Modells mit `.evaluate` evaluieren oder mit `.predict` Vorhersagen treffen und den MSE mit der sklearn-Implementierung `mean_squared_error` berechnen. Beide Wege sollten denselben Wert liefern.

`.evaluate` ist praktisch, weil es die Metriken mit genau der Vorverarbeitungspipeline berechnet, die im Modell enthalten ist. Ein manueller Aufruf von `.predict` + `mean_squared_error` sollte dasselbe ergeben und ist eine gute Plausibilitätsprüfung dafür, dass kein Data Leakage aufgetreten ist.

In [ ]:
# Kleines Modell mit .evaluate auf dem Testdatensatz evaluieren
loss, mse = small_model.evaluate(X_test, y_test, verbose=2)
evaluation_results["small"] = mse
print(f"Model MSE: {mse}")
print("--------" * 5)

# Werte für den Testdatensatz vorhersagen
y_pred = small_model.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))

**Erklärung des Ergebnisses.** Das kleine Modell erreicht in diesem Durchlauf einen Test-MSE von etwa **9,97**. Das ist unsere Baseline. Behalte sie im Kopf: Schon ein bewusst klein gehaltenes Netz liefert auf diesem Datensatz ordentliche Ergebnisse. Das deutet darauf hin, dass das Problem kein großes Modell braucht.

### Checkpoints während des Trainings speichern

Das Training eines neuronalen Netzes kann lange dauern, deshalb hilft es, das Modell schon während des Trainings zu speichern. Der Callback `tf.keras.callbacks.ModelCheckpoint` übernimmt das: Er schreibt das Modell (oder nur seine Gewichte) während und am Ende des Trainings auf die Festplatte. So kannst du ein trainiertes Modell wiederverwenden, ohne es neu zu trainieren, oder das Training fortsetzen, falls es unterbrochen wurde.

Ein **Callback** ist ein Objekt, das du an `model.fit` übergibst und das Keras an festgelegten Punkten des Trainings automatisch ausführt (zum Beispiel am Ende jeder Epoche). Der Checkpoint-Callback unten überwacht den Durchlauf und speichert die Gewichte, während das Training läuft. Im nächsten Abschnitt setzen wir ihn beim mittleren Modell ein.

Die hier verwendeten Optionen:

- `save_weights_only=True` speichert nur die Gewichte, nicht den vollständigen Modellgraphen, und hält so die Dateien klein. Die Architektur erstellst du aus dem Code neu und lädst die Gewichte neu.
- `verbose=0` unterdrückt die Ausgabe. Setz es auf `1`, damit du jedes Mal eine Meldung siehst, wenn der Callback speichert.

In [ ]:
# Pfad definieren, unter dem die Checkpoints gespeichert werden sollen
checkpoint_path = "training_1/cp.weights.h5"
checkpoint_dir = os.path.dirname(checkpoint_path)

# Callback erstellen, der die Gewichte des Modells speichert
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path, save_weights_only=True, verbose=0
)  # verbose != 0 setzen, wenn du während des Trainings eine Ausgabe willst

### Mittleres Modell

Wir erhöhen die Kapazität, indem wir eine dritte verborgene Schicht hinzufügen und 64 Neuronen pro Schicht verwenden. Die Schritte zum Kompilieren und Trainieren sind dieselben wie zuvor, nur übergeben wir diesmal den Checkpoint-Callback, den wir gerade definiert haben.

Mehr Kapazität kann Underfitting verringern, aber das Modell trainiert langsamer und kann anfangen, Overfitting zu zeigen. Vergleich die Kurven und den Test-MSE des mittleren Modells mit dem kleinen Modell, um zu sehen, ob die zusätzlichen Parameter auf diesem Datensatz wirklich helfen.

In [ ]:
# Funktion für die Architektur des mittleren Modells
def get_compiled_medium_model():
    medium_model = tf.keras.Sequential(
        [
            tf.keras.Input(shape=(12,)),
            tf.keras.layers.Dense(
                64, kernel_initializer="uniform", activation="relu"
            ),
            tf.keras.layers.Dense(64, kernel_initializer="uniform", activation="relu"),
            tf.keras.layers.Dense(64, kernel_initializer="uniform", activation="relu"),
            tf.keras.layers.Dense(1, kernel_initializer="uniform"),
        ]
    )

    medium_model.compile(optimizer=make_optimizer(), loss="mae", metrics=["mse"])
    return medium_model

In [ ]:
# Zusammenfassung des mittleren Modells
with tf.device("/cpu:0"):
    medium_model = get_compiled_medium_model()
    print(medium_model.summary())

In [ ]:
# Mittleres Modell mit dem neuen Callback trainieren
with tf.device("/cpu:0"):
    training_history["medium"] = medium_model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        verbose=0,
        steps_per_epoch=STEPS_PER_EPOCH,
        epochs=EPOCHS,
        callbacks=[cp_callback],
    )  # Callback an das Training übergeben

In [ ]:
# MSE des mittleren Modells visualisieren
plot_metric(training_history["medium"])

In [ ]:
# Verlust des mittleren Modells visualisieren
plot_loss(training_history["medium"])

In [ ]:
# Mittleres Modell mit .evaluate auf dem Testdatensatz evaluieren
loss, mse = medium_model.evaluate(X_test, y_test, verbose=2)
evaluation_results["medium"] = mse
print(f"Model MSE: {mse}")

**Erklärung des Ergebnisses.** Das mittlere Modell erreicht in diesem Durchlauf einen Test-MSE von etwa **11,72**. Das ist etwas schlechter als die 9,97 des kleinen Modells, nicht besser. Auf einem kleinen Datensatz wie Boston Housing verbessert zusätzliche Kapazität die Generalisierung nicht automatisch: Die zusätzlichen Parameter geben dem Netz mehr Spielraum, sich an Rauschen anzupassen. Genau für diese Art von Vergleich ist der Testdatensatz da.

### Großes Modell

Jetzt erhöhen wir die Kapazität deutlich: vier verborgene Schichten mit je 512 Neuronen. Ein so großes Netz kann den Trainingsdatensatz schnell auswendig lernen. Wir erwarten also, dass der Trainingsverlust schnell sinkt, während der Validierungsverlust abflacht oder steigt. Weil wir für alle Größen denselben Lernratenplan verwenden, bleibt der Vergleich fair.

In [ ]:
# Funktion für die Architektur des großen Modells
def get_compiled_large_model():
    large_model = tf.keras.Sequential(
        [
            tf.keras.Input(shape=(12,)),
            tf.keras.layers.Dense(
                512, kernel_initializer="uniform", activation="relu"
            ),
            tf.keras.layers.Dense(512, kernel_initializer="uniform", activation="relu"),
            tf.keras.layers.Dense(512, kernel_initializer="uniform", activation="relu"),
            tf.keras.layers.Dense(512, kernel_initializer="uniform", activation="relu"),
            tf.keras.layers.Dense(1, kernel_initializer="uniform"),
        ]
    )

    large_model.compile(optimizer=make_optimizer(), loss="mae", metrics=["mse"])
    return large_model

In [ ]:
# Zusammenfassung des großen Modells
with tf.device("/cpu:0"):
    large_model = get_compiled_large_model()
    print(large_model.summary())

In [ ]:
# Großes Modell trainieren
with tf.device("/cpu:0"):
    training_history["large"] = large_model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        verbose=0,
        steps_per_epoch=STEPS_PER_EPOCH,
        epochs=EPOCHS,
    )

In [ ]:
# MSE des großen Modells visualisieren
plot_metric(training_history["large"])

In [ ]:
# Verlust des großen Modells visualisieren
plot_loss(training_history["large"])

In [ ]:
# Großes Modell mit .evaluate auf dem Testdatensatz evaluieren
loss, mse = large_model.evaluate(X_test, y_test, verbose=2)
evaluation_results["large"] = mse
print(f"Model MSE: {mse}")

**Erklärung des Ergebnisses.** Das große Modell erreicht in diesem Durchlauf einen Test-MSE von etwa **11,60**, ungefähr so viel wie das mittlere Modell und immer noch schlechter als das kleine. Obwohl es weit mehr Parameter hat, generalisiert es hier nicht besser. Das motiviert den nächsten Abschnitt: Statt Kapazität hinzuzufügen, ergänzen wir ein großes Modell um **Regularisierung**, damit es besser generalisiert.

## Strategien gegen Overfitting

Wenn ein Modell Overfitting zeigt, hast du ein paar Möglichkeiten. Das Diagramm unten fasst zusammen, wie du die Kurven liest und was jeweils zu tun ist.

```mermaid
flowchart TD
    A[Trainings- vs Validierungskurven ansehen] --> B{Beide Fehler hoch?}
    B -->|Ja| C[Underfitting: Kapazität erhöhen oder länger trainieren]
    B -->|Nein| D{Validierungsfehler schlechter als Trainingsfehler?}
    D -->|Ja| E[Overfitting: mehr Daten hinzufügen, Kapazität verringern<br/>oder Regularisierung L1/L2/Dropout hinzufügen]
    D -->|Nein| F[Gute Anpassung: dieses Modell behalten]
```

### Regularisierung

Vielleicht kennst du das Prinzip von Ockhams Rasiermesser: Gibt es für etwas zwei Erklärungen, ist am wahrscheinlichsten die „einfachste“ richtig, also die mit den wenigsten Annahmen. Das gilt auch für die Modelle, die neuronale Netze lernen: Für bestimmte Trainingsdaten und eine Netzarchitektur gibt es mehrere Sätze von Gewichtswerten (mehrere Modelle), die die Daten erklären könnten, und einfachere Modelle neigen weniger zu Overfitting als komplexe.

Im nächsten Abschnitt probieren wir eine Form der **Regularisierung** aus. Hoffentlich kennst du die L1- und L2-Regularisierung schon, da wir im Bootcamp mehrmals darüber gesprochen haben. Trotzdem frischen wir die allgemeinen Konzepte auf, bevor wir weitermachen:

Wir können Regularisierungstechniken anwenden, um ein einfacheres Modell zu erzeugen und so Overfitting zu verhindern. Ein „einfaches Modell“ ist in diesem Zusammenhang ein Modell, bei dem die Verteilung der Parameterwerte eine geringere Entropie hat (oder ein Modell mit insgesamt weniger Parametern). Eine gängige Methode gegen Overfitting ist es daher, die Komplexität eines Netzes einzuschränken, indem wir seine Gewichte zwingen, nur kleine Werte anzunehmen. Dadurch wird die Verteilung der Gewichtswerte „regulärer“. Dazu werden der Verlustfunktion des Netzes Kosten für große Gewichte hinzugefügt. Diese Kosten gibt es in zwei Varianten:

* [L1-Regularisierung](https://developers.google.com/machine-learning/glossary/#L1_regularization), bei der die hinzugefügten Kosten proportional zum Absolutwert der Gewichtskoeffizienten sind (d. h. zur sogenannten „L1-Norm“ der Gewichte).

* [L2-Regularisierung](https://developers.google.com/machine-learning/glossary/#L2_regularization), bei der die hinzugefügten Kosten proportional zum Quadrat des Werts der Gewichtskoeffizienten sind (d. h. zur sogenannten quadrierten „L2-Norm“ der Gewichte). Mit einfachem SGD entspricht das dem Weight Decay mit passenden Koeffizienten. Bei adaptiven Optimierern wie Adam unterscheidet sich ein L2-Strafterm im Verlust vom entkoppelten Weight Decay, wie ihn [AdamW](https://keras.io/api/optimizers/adamw/) verwendet.

Die L1-Regularisierung drängt Gewichte auf genau null und fördert so ein dünnbesetztes Modell. Die L2-Regularisierung bestraft die Gewichtsparameter, ohne sie dünnbesetzt zu machen, da der Strafterm für kleine Gewichte gegen null geht: ein Grund, warum L2 verbreiteter ist.

In `tf.keras` fügst du Gewichtsregularisierung hinzu, indem du den Schichten Instanzen von Gewichtsregularisierern als Schlüsselwortargumente übergibst. Fügen wir unserem großen Modell von oben eine L2-Gewichtsregularisierung hinzu:

In [ ]:
# Großes Modell mit zusätzlicher L2-Regularisierung
def get_compiled_l2_model():
    l2_model = tf.keras.Sequential(
        [
            tf.keras.Input(shape=(12,)),
            tf.keras.layers.Dense(
                512,
                kernel_initializer="uniform",
                activation="relu",
                kernel_regularizer=regularizers.l2(0.01),
            ),
            tf.keras.layers.Dense(
                512,
                kernel_initializer="uniform",
                activation="relu",
                kernel_regularizer=regularizers.l2(0.01),
            ),
            tf.keras.layers.Dense(
                512,
                kernel_initializer="uniform",
                activation="relu",
                kernel_regularizer=regularizers.l2(0.01),
            ),
            tf.keras.layers.Dense(
                512,
                kernel_initializer="uniform",
                activation="relu",
                kernel_regularizer=regularizers.l2(0.01),
            ),
            tf.keras.layers.Dense(1, kernel_initializer="uniform"),
        ]
    )

    l2_model.compile(optimizer=make_optimizer(), loss="mae", metrics=["mse"])
    return l2_model

`l2(0.01)` bedeutet, dass jeder Koeffizient in der Gewichtsmatrix der Schicht `0.01 * weight_coefficient_value**2` zum gesamten **Verlust** des Netzes addiert.

Wenn du weiterhin Overfitting siehst, erhöh den Koeffizienten (zum Beispiel auf `l2(0.02)`). Wenn das Training stagniert, verringere ihn. Der L2-Term wird pro Schicht angewendet. Du kannst also nur die dichten Schichten regularisieren, die die meisten Parameter tragen, statt alle Schichten.

In [ ]:
# Zusammenfassung der Modellarchitektur
with tf.device("/cpu:0"):
    l2_model = get_compiled_l2_model()
    print(l2_model.summary())

In [ ]:
# Regularisiertes Modell trainieren
with tf.device("/cpu:0"):
    training_history["l2"] = l2_model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        verbose=0,
        steps_per_epoch=STEPS_PER_EPOCH,
        epochs=EPOCHS,
    )

In [ ]:
# MSE des L2-Modells visualisieren
plot_metric(training_history["l2"])

In [ ]:
# Verlust des L2-Modells visualisieren
plot_loss(training_history["l2"])

In [ ]:
# Regularisiertes Modell evaluieren
loss, mse = l2_model.evaluate(X_test, y_test, verbose=2)
evaluation_results["L2"] = mse
print(f"Model MSE: {mse}")

**Erklärung des Ergebnisses.** Das L2-Modell erreicht in diesem Durchlauf einen Test-MSE von etwa **8,07**, das beste Ergebnis aller bisherigen Modelle (besser als die 9,97 des kleinen Modells und deutlich unter den 11,60 des nicht regularisierten großen Modells). Die Architektur ist dieselbe wie beim großen Modell, aber der L2-Strafterm hält die Gewichte klein. Das bremst das Overfitting und verbessert die Generalisierung.

L2 zum großen Modell hinzuzufügen, hilft also deutlich: Es macht aus der Architektur, die am schlechtesten generalisiert, hier die mit der besten Leistung, obwohl die Anzahl der Parameter genau gleich bleibt. Darin liegt der Wert der Regularisierung: Du kannst ein Modell mit hoher Kapazität behalten und gleichzeitig seine Neigung zu Overfitting verringern.

### Dropout

Dropout ist eine der wirksamsten und am weitesten verbreiteten Regularisierungstechniken für neuronale Netze. Entwickelt wurde es von Hinton und seinen Studierenden an der University of Toronto.

Die Intuition dahinter: Weil sich ein Knoten nicht darauf verlassen kann, dass ein bestimmter anderer Knoten vorhanden ist, muss jeder Knoten Features lernen, die für sich allein nützlich sind. Während des Trainings setzt Dropout zufällig einen Anteil der Ausgaben einer Schicht auf null. Zum Beispiel könnte das Maskieren von `[0.2, 0.5, 1.3, 0.8, 1.1]` vor dem Neuskalieren `[0, 0.5, 1.3, 0, 1.1]` ergeben.

Die **Dropout-Rate** ist die Wahrscheinlichkeit, mit der jede Ausgabe auf null gesetzt wird, üblicherweise zwischen 0,2 und 0,5. [Keras Dropout](https://keras.io/api/layers/regularization_layers/dropout/) skaliert die beibehaltenen Ausgaben während des Trainings mit `1 / (1 - rate)`, damit ihre Erwartungswerte erhalten bleiben. Bei der Inferenz wird nichts verworfen oder neu skaliert.

In `tf.keras` fügst du Dropout mit der `Dropout`-Schicht hinzu, die du hinter die Schicht setzt, deren Ausgabe du verwerfen willst. Unten fügen wir nach jeder dichten Schicht eine Dropout-Schicht ein. Dropout verlangsamt die Konvergenz meist etwas, weil das Netz in jedem Batch eine verrauschtere Version von sich selbst sieht, was zu erwarten ist. Du kannst Raten wie 0,2, 0,3 und 0,5 ausprobieren, um die Regularisierungsstärke gegen die Trainingsgeschwindigkeit abzuwägen.

In [ ]:
# Großes Modell mit zusätzlicher Dropout-Schicht
def get_compiled_dropout_model():
    dropout_model = tf.keras.Sequential(
        [
            tf.keras.Input(shape=(12,)),
            tf.keras.layers.Dense(
                512, kernel_initializer="uniform", activation="relu"
            ),
            tf.keras.layers.Dropout(0.25),
            tf.keras.layers.Dense(512, kernel_initializer="uniform", activation="relu"),
            tf.keras.layers.Dropout(0.25),
            tf.keras.layers.Dense(512, kernel_initializer="uniform", activation="relu"),
            tf.keras.layers.Dropout(0.25),
            tf.keras.layers.Dense(512, kernel_initializer="uniform", activation="relu"),
            tf.keras.layers.Dropout(0.25),
            tf.keras.layers.Dense(1, kernel_initializer="uniform"),
        ]
    )

    dropout_model.compile(optimizer=make_optimizer(), loss="mae", metrics=["mse"])
    return dropout_model

In [ ]:
# Zusammenfassung der Modellarchitektur
with tf.device("/cpu:0"):
    dropout_model = get_compiled_dropout_model()
    print(dropout_model.summary())

In [ ]:
# Dropout-Modell trainieren
with tf.device("/cpu:0"):
    training_history["dropout"] = dropout_model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        verbose=0,
        steps_per_epoch=STEPS_PER_EPOCH,
        epochs=EPOCHS,
    )

In [ ]:
# MSE des Dropout-Modells visualisieren
plot_metric(training_history["dropout"])

In [ ]:
# Verlust des Dropout-Modells visualisieren
plot_loss(training_history["dropout"])

In [ ]:
# Dropout-Modell evaluieren
loss, mse = dropout_model.evaluate(X_test, y_test, verbose=2)
evaluation_results["dropout"] = mse
print(f"Model MSE: {mse}")

**Erklärung des Ergebnisses.** Das Dropout-Modell erreicht in diesem Durchlauf einen Test-MSE von etwa **12,35**, das schlechteste Ergebnis der Gruppe. Das heißt nicht, dass Dropout versagt. Es zeigt vielmehr, dass dieser Datensatz bei dieser Modellgröße kaum Overfitting zeigt: Das Verwerfen von Neuronen fügt vor allem Rauschen hinzu, ohne dass es ein echtes Overfitting-Problem zu lösen gibt. Dropout lohnt sich am meisten, wenn ein Modell auf einem größeren, verrauschteren Datensatz wirklich Overfitting zeigt.

### L2 und Dropout kombinieren

Es ist üblich, Regularisierungstechniken in einem Modell zu kombinieren. L2 hält die Gewichte klein, während Dropout die Koadaptation zwischen Neuronen verringert. Ein sinnvoller Ausgangspunkt ist die Dropout-Rate, die du oben gefunden hast, zusammen mit einem etwas kleineren L2-Koeffizienten, damit das Modell nicht überregularisiert wird.

In [ ]:
# Großes Modell mit zusätzlicher Dropout-Schicht und L2-Regularisierung
def get_compiled_l2_dropout_model():
    l2_dropout_model = tf.keras.Sequential(
        [
            tf.keras.Input(shape=(12,)),
            tf.keras.layers.Dense(
                512,
                kernel_initializer="uniform",
                activation="relu",
                kernel_regularizer=regularizers.l2(0.01),
            ),
            tf.keras.layers.Dropout(0.25),
            tf.keras.layers.Dense(
                512,
                kernel_initializer="uniform",
                activation="relu",
                kernel_regularizer=regularizers.l2(0.01),
            ),
            tf.keras.layers.Dropout(0.25),
            tf.keras.layers.Dense(
                512,
                kernel_initializer="uniform",
                activation="relu",
                kernel_regularizer=regularizers.l2(0.01),
            ),
            tf.keras.layers.Dropout(0.25),
            tf.keras.layers.Dense(
                512,
                kernel_initializer="uniform",
                activation="relu",
                kernel_regularizer=regularizers.l2(0.01),
            ),
            tf.keras.layers.Dropout(0.25),
            tf.keras.layers.Dense(1, kernel_initializer="uniform"),
        ]
    )

    l2_dropout_model.compile(optimizer=make_optimizer(), loss="mae", metrics=["mse"])
    return l2_dropout_model

In [ ]:
# Zusammenfassung der Modellarchitektur
with tf.device("/cpu:0"):
    l2_dropout_model = get_compiled_l2_dropout_model()
    print(l2_dropout_model.summary())

In [ ]:
# Modell trainieren
with tf.device("/cpu:0"):
    training_history["l2_dropout"] = l2_dropout_model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        verbose=0,
        steps_per_epoch=STEPS_PER_EPOCH,
        epochs=EPOCHS,
    )

In [ ]:
# MSE des Dropout-L2-Modells visualisieren
plot_metric(training_history["l2_dropout"])

In [ ]:
# Verlust des Dropout-L2-Modells visualisieren
plot_loss(training_history["l2_dropout"])

In [ ]:
# Dropout-L2-Modell evaluieren
loss, mse = l2_dropout_model.evaluate(X_test, y_test, verbose=2)
evaluation_results["L2 + dropout"] = mse
print(f"Model MSE: {mse}")

**Erklärung des Ergebnisses.** Das kombinierte Modell aus L2 und Dropout erreicht in diesem Durchlauf einen Test-MSE von etwa **11,17**. Das ist besser als Dropout allein, aber nicht so gut wie L2 allein (8,07). Zwei Regularisierer gleichzeitig auf einen Datensatz anzuwenden, der kaum Overfitting zeigt, schränkt das Modell zu stark ein. Deshalb gewinnt hier oft eine einzelne, gut gewählte Technik.

## Ergebnisse visualisieren

Mit `tfdocs.plots.HistoryPlotter` können wir die Trainingsverläufe überlagern und alle Modelle in einem Diagramm vergleichen.

Übergib das Dictionary `training_history` an `HistoryPlotter.plot`. Die Hilfsfunktion wandelt jeden gespeicherten Verlauf in einen Pandas-DataFrame um, trägt die Epochen auf der x-Achse ab und verwendet die Schlüssel des Dictionarys als Legende. So kannst du weitere Experimente hinzufügen, ohne den Code für die Darstellung zu ändern.

In [ ]:
# HistoryPlotter instanziieren
plotter = tfdocs.plots.HistoryPlotter(metric="mse", smoothing_std=10)

In [ ]:
# Ergebnisse aller Modelle darstellen
plotter.plot(training_history)
plt.xlim([5, max(plt.xlim())])
plt.ylim([0, 50])
plt.xlabel("Epochs")
plt.title("Comparison of all models")
plt.legend(loc="upper right");

### Modelle vergleichen

Das Diagramm zeigt, dass die Kurven nah beieinanderliegen: Auf diesem Datensatz verhalten sich die Modelle recht ähnlich, und keines zeigt dramatisches Overfitting.

Um sie genau zu vergleichen, sammeln wir beim Evaluieren den Test-MSE jedes Modells im Dictionary `evaluation_results` (analog zum Dictionary `training_history` für die Trainingskurven). Die Zelle unten macht aus diesem Dictionary eine Tabelle, sortiert vom besten (niedrigster MSE) bis zum schlechtesten Modell.

In diesem Durchlauf fallen zwei Dinge auf. Erstens ist das **L2-Modell das beste**, und das Netz einfach größer zu machen (mittel, groß) hat die kleine Baseline **nicht** geschlagen: Mehr Kapazität ist nicht automatisch besser. Zweitens brachte die Regularisierung eines großen Modells mit L2 die größte Verbesserung.

> [!NOTE]
> Nur die Aufteilung in Trainings- und Testdaten hat einen festen Seed, nicht die Initialisierung der Gewichte oder das Dropout. Deshalb verschieben sich die genauen MSE-Werte von Durchlauf zu Durchlauf, und die Reihenfolge der mittleren Modelle kann sich ändern. Betrachte die Zahlen als Richtwerte und konzentriere dich auf das Muster: Dieser kleine Datensatz zeigt kaum Overfitting, daher bringt zusätzliche Kapazität wenig, und ein einzelner L2-Strafterm hilft am meisten.

In [ ]:
# Test-MSE aller Modelle in einer Tabelle sammeln, sortiert vom besten (niedrigsten) bis zum schlechtesten
results_df = pd.DataFrame({"test_mse": evaluation_results}).sort_values("test_mse")
results_df

In [ ]:
print("Large Model loss and MSE:")
loss_large, mse_large = large_model.evaluate(X_test, y_test, verbose=2)
print("----" * 10)
print("L2 Model loss and MSE:")
loss_l2, mse_l2 = l2_model.evaluate(X_test, y_test, verbose=2)

## Zusammenfassung

In diesem Notebook hast du:

- Overfitting und Underfitting diagnostiziert, indem du Trainings- und Validierungskurven gelesen hast.
- kleine, mittlere und große Netze auf dem Datensatz Boston Housing erstellt und verglichen.
- L2-Regularisierung und Dropout (und beides zusammen) auf ein großes Modell angewendet.
- jedes Modell auf dem zurückgehaltenen Testdatensatz evaluiert und die Ergebnisse in einer Tabelle verglichen.

Der Datensatz Boston Housing ist klein und zeigt bei diesen Modellgrößen kaum Overfitting. Deshalb sind die Unterschiede gering und etwas verrauscht. Zusätzliche Kapazität hat nicht geholfen, während ein einzelner L2-Strafterm den besten Test-MSE brachte. Die gängigsten Wege, Overfitting zu verhindern, sind:

- mehr Trainingsdaten beschaffen.
- die Kapazität des Netzes verringern.
- Gewichtsregularisierung (L1 oder L2) hinzufügen.
- Dropout hinzufügen.

Zwei weitere Ansätze, die hier nicht behandelt werden, sind Datenaugmentierung und Batch-Normalisierung. Jede Technik kann für sich allein helfen, und eine Kombination kann manchmal noch mehr helfen, auch wenn das, wie wir gesehen haben, auf einem Datensatz mit kaum Overfitting nicht garantiert ist.

## Unser Modell speichern

Ruf `model.save` auf, um die Architektur, die Gewichte und die Trainingskonfiguration eines Modells in einer einzigen Datei zu speichern. So kannst du ein Modell exportieren und ohne den ursprünglichen Python-Code wiederverwenden. (Benutzerdefinierte Objekte, etwa als Unterklasse definierte Modelle oder Schichten, brauchen beim Speichern und Laden eine besondere Behandlung.) Da auch der Zustand des Optimierers gespeichert wird, kannst du das Training dort fortsetzen, wo du aufgehört hast.

Ein Modell lässt sich in zwei Formaten speichern: im `SavedModel`-Format von TensorFlow (Standard in TF2.x) und in `HDF5`. Ein gespeichertes Modell ist portabel: Du kannst es in TensorFlow.js laden, um es im Browser auszuführen, oder es mit TensorFlow Lite konvertieren, um es auf mobilen Geräten auszuführen.

Die Zelle unten speichert das große Modell unter `saved_model/my_large_model.keras`. Wenn du sie erneut ausführst, wird diese Datei überschrieben.

### SavedModel-Format

Das `SavedModel`-Format ist eine weitere Möglichkeit, Modelle zu serialisieren. In diesem Format gespeicherte Modelle lassen sich mit `tf.keras.models.load_mode` wiederherstellen und sind mit TensorFlow Serving kompatibel.

In der nächsten Zelle speichern wir unser großes Modell in diesem Format.

In [ ]:
# Gesamtes großes Modell als SavedModel speichern.
!mkdir -p saved_model
large_model.save("saved_model/my_large_model.keras")

Wenn du sehen willst, wie du das Modell laden kannst, um Vorhersagen zu treffen, sieh dir das nächste Notebook an ...